In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

from langchain.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)

# グラフの状態
class OverAllState(TypedDict):
    topic: str
    joke: str
    improved_joke: str
    final_joke: str

# ノード
def generate_joke(state: OverAllState) -> OverAllState:
    """1回目の大規模言語モデル呼び出し。初期のジョークを生成する"""

    msg = model.invoke(
        [HumanMessage(content=f"「{state['topic']}」についての短いジョークを書いてください")]
    )
    return {"joke": msg.content}

def check_punchline(state: OverAllState) -> Literal["pass", "no_pass"]:
    """ジョークにオチが含まれているかどうかを判定する"""

    # 簡易判定：ジョークに疑問符または感嘆符が含まれているか
    if "?" in state["joke"] or "!" in state["joke"]:
        return "pass"
    return "no_pass"

def improve_joke(state: OverAllState) -> OverAllState:
    """2回目の大規模言語モデル呼び出し。地口（だじゃれ）を加えてジョークを改善する"""

    msg = model.invoke(
        [HumanMessage(content=f"地口（だじゃれ）を加えて、以下のジョークをもっと面白くしてください：\n{state['joke']}")]
    )
    return {"improved_joke": msg.content}

def polish_joke(state: OverAllState) -> OverAllState:
    """3回目の大規模言語モデル呼び出し。ジョークを最終的に仕上げる"""

    msg = model.invoke(
        [HumanMessage(content=f"以下のジョークに意外などんでん返しを加えてください：\n{state['improved_joke']}")]
    )
    return {"final_joke": msg.content}

# ワークフローを構築
builder = StateGraph(state_schema=OverAllState)

# ノードを追加
builder.add_node("generate_joke", generate_joke)
builder.add_node("improve_joke", improve_joke)
builder.add_node("polish_joke", polish_joke)

# エッジを追加して各ノードを接続
builder.add_edge(START, "generate_joke")
builder.add_conditional_edges(
    "generate_joke",
    check_punchline,
    {
        "no_pass": "improve_joke",
        "pass": END,
    },
)
builder.add_edge("improve_joke", "polish_joke")
builder.add_edge("polish_joke", END)

# ワークフローをコンパイル
graph = builder.compile()

# ワークフローを呼び出す
response = graph.invoke({"topic": "猫"})
print(response)

# トポロジー構造を描画
from IPython.display import display
display(graph)